In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import f1_score


In [2]:
df = pd.read_csv('../Data/final_dataset.csv')

In [3]:
y = df['target']
x = df.drop('target', axis=1) 

In [4]:
train = df[df['datetime'] < '2015-06-01']
test  = df[df['datetime'] >= '2015-06-01']

drop_cols = ['target', 'failure_flag', 'datetime', 'last_maint_datetime']

X_train = train.drop(columns=drop_cols)
y_train = train['target']

X_test = test.drop(columns=drop_cols)
y_test = test['target']

In [5]:
cat_features = ['comp', 'model']
num_features = [col for col in X_train.columns if col not in cat_features]

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('num', 'passthrough', num_features)
    ]
)

In [6]:
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 1075.8392857142858


In [7]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)
model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', xgb)
])
model.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['comp', 'model']),
                                                 ('num', 'passthrough',
                                                  ['machineID', 'volt',
                                                   'rotate', 'pressure',
                                                   'vibration', 'error_count',
                                                   'maint_flag',
                                                   'days_since_maint',
                                                   'age'])])),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, cols...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=300, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [23]:
y_risk  = model.predict_proba(X_test)[:,1]
df_results = X_test.copy()
df_results['risk_score'] = y_risk
df_results['risk_percent'] = df_results['risk_score'] * 100


In [24]:
def risk_level(p):
    if p < 0.2:
        return "LOW 🟢"
    elif p < 0.6:
        return "MEDIUM 🟡"
    else:
        return "HIGH 🔴"

df_results['risk_level'] = df_results['risk_score'].apply(risk_level)

In [27]:
df_results[['risk_percent', 'risk_level']].sort_values('risk_percent', ascending=False).head(10)

,machineID,volt,rotate,pressure,vibration,error_count,comp,maint_flag,days_since_maint,model,age,risk_score,risk_percent,risk_level
659171,43,170.766946,419.651357,99.854886,53.095003,0.0,comp2,1,14,model3,14,0.987772,98.777184,HIGH 🔴
563132,71,162.198684,383.112684,130.729617,48.115181,0.0,comp1,1,14,model2,18,0.987681,98.768105,HIGH 🔴
857342,56,215.719617,485.034848,126.248526,39.794773,0.0,comp2,1,14,model1,10,0.987581,98.758057,HIGH 🔴
428395,54,172.984626,418.259785,138.988900,40.347904,0.0,comp4,1,14,model2,10,0.986607,98.660660,HIGH 🔴
642421,56,171.306803,338.920302,118.306803,39.391049,0.0,comp1,1,14,model1,10,0.986166,98.616600,HIGH 🔴
699754,94,155.690116,435.629004,126.004586,45.532706,0.0,comp4,1,14,model2,18,0.985780,98.578041,HIGH 🔴
611495,67,168.515283,309.885241,106.000885,35.967416,0.0,comp4,1,14,model4,14,0.985101,98.510078,HIGH 🔴
770949,94,155.151608,470.283342,125.780802,51.089939,0.0,comp3,1,14,model2,18,0.985063,98.506287,HIGH 🔴
838049,42,167.392218,412.195837,131.278338,37.844802,0.0,comp1,1,14,model1,7,0.984982,98.498230,HIGH 🔴
499543,49,176.860550,420.055947,137.452436,40.578126,0.0,comp4,1,14,model1,15,0.984658,98.465797,HIGH 🔴
